# Spaceship Titanic — Step 2: Baselines + Feature Ablation

Plan, in order:
1. Two baselines: naive majority-class, and a "CryoSleep rule" baseline (predict the per-CryoSleep majority class).
2. Feature ablation: one fixed, untuned logistic regression, run across feature subsets, on a single held-out validation split. This isolates which features matter without hyperparameter tuning confounding the comparison.
3. Take the best feature subset and grid search hyperparameters across two model families.

We use a single stratified train/validation split for every experiment in this notebook so comparisons are apples-to-apples.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from src.features import engineer_features, SPEND_COLS

pd.set_option("display.max_columns", None)

raw_train = pd.read_csv("../data/train.csv")
raw_test = pd.read_csv("../data/test.csv")

train, test = engineer_features(raw_train, raw_test)
print(train.shape, test.shape)
print("Any missing left in train?", train.isna().sum().sum())
train.head()

(8693, 22) (4277, 21)
Any missing left in train? 399


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Group,GroupSize,Deck,CabinNum,Side,TotalSpend,IsSolo,GroupSizeBin,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,0001,1,B,0.0,P,0.0,True,solo,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,0002,1,F,0.0,S,736.0,True,solo,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,0003,2,A,0.0,S,10383.0,False,small,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,0003,2,A,0.0,S,5176.0,False,small,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,0004,1,F,1.0,S,1091.0,True,solo,True


In [2]:
train_df, val_df = train_test_split(
    train, test_size=0.2, stratify=train["Transported"], random_state=42
)
print("Train:", train_df.shape, "Val:", val_df.shape)

# Baseline 1: naive majority class
naive_pred = pd.Series(train_df["Transported"].mode().iat[0], index=val_df.index)
naive_acc = accuracy_score(val_df["Transported"], naive_pred)
print(f"\nBaseline 1 (naive majority class): {naive_acc:.4f}")

# Baseline 2: per-CryoSleep majority class
cryo_majority = train_df.groupby("CryoSleep")["Transported"].agg(lambda s: s.mode().iat[0])
cryo_pred = val_df["CryoSleep"].map(cryo_majority)
cryo_acc = accuracy_score(val_df["Transported"], cryo_pred)
print(f"Baseline 2 (CryoSleep-rule): {cryo_acc:.4f}")

Train: (6954, 22) Val: (1739, 22)

Baseline 1 (naive majority class): 0.5037
Baseline 2 (CryoSleep-rule): 0.7131


## Feature ablation (fixed model, no tuning)

One untuned `LogisticRegression` (default `C=1.0`, `max_iter=1000`), run across feature subsets we care about: just CryoSleep, just HomePlanet, just Cabin Deck, just GroupSize, then combinations, then everything we've engineered. Categorical columns get one-hot encoded, numeric columns get standard-scaled, inside a `ColumnTransformer` so each subset is handled correctly regardless of dtype mix.

In [3]:
ALL_CATEGORICAL = ["HomePlanet", "Destination", "Deck", "Side", "GroupSizeBin"]
ALL_NUMERIC = ["Age", "TotalSpend", "CabinNum", "GroupSize"] + SPEND_COLS
ALL_BOOLEAN = ["CryoSleep", "VIP", "IsSolo"]


def build_pipeline(categorical, numeric, boolean):
    transformers = []
    if categorical:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical))
    if numeric:
        transformers.append(("num", StandardScaler(), numeric))
    if boolean:
        transformers.append(("bool", "passthrough", boolean))
    preprocessor = ColumnTransformer(transformers)
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])


def evaluate_feature_set(name, categorical, numeric, boolean):
    cols = categorical + numeric + boolean
    pipe = build_pipeline(categorical, numeric, boolean)
    pipe.fit(train_df[cols], train_df["Transported"])
    preds = pipe.predict(val_df[cols])
    acc = accuracy_score(val_df["Transported"], preds)
    return {"Experiment": name, "Features": cols, "Accuracy": acc}


feature_sets = [
    ("CryoSleep only", [], [], ["CryoSleep"]),
    ("HomePlanet only", ["HomePlanet"], [], []),
    ("Cabin Deck only", ["Deck"], [], []),
    ("GroupSize only", [], [], ["IsSolo"]),
    ("CryoSleep + HomePlanet", ["HomePlanet"], [], ["CryoSleep"]),
    ("CryoSleep + Cabin Deck", ["Deck"], [], ["CryoSleep"]),
    ("CryoSleep + HomePlanet + Cabin Deck", ["HomePlanet", "Deck"], [], ["CryoSleep"]),
    ("CryoSleep + HomePlanet + Deck + GroupSize", ["HomePlanet", "Deck"], [], ["CryoSleep", "IsSolo"]),
    ("All engineered features", ALL_CATEGORICAL, ALL_NUMERIC, ALL_BOOLEAN),
]

results = [evaluate_feature_set(*fs) for fs in feature_sets]
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
results_df[["Experiment", "Accuracy"]]

,Experiment,Accuracy
8,All engineered features,0.791834
0,CryoSleep only,0.713053
4,CryoSleep + HomePlanet,0.713053
7,CryoSleep + HomePlanet + Deck + GroupSize,0.709603
5,CryoSleep + Cabin Deck,0.705578
6,CryoSleep + HomePlanet + Cabin Deck,0.704428
1,HomePlanet only,0.588844
2,Cabin Deck only,0.579643
3,GroupSize only,0.546866


## Isolating the numeric features

Test whether the jump to 79% is really coming from the continuous spend/Age features rather than "more features in general." We compare: numerics alone (no CryoSleep, no categoricals), CryoSleep + numerics, and categoricals + numerics, to triangulate where the signal actually lives.

In [4]:
NUMERIC_SPEND = ["TotalSpend"] + SPEND_COLS
NUMERIC_ALL = ALL_NUMERIC  # Age, TotalSpend, CabinNum, GroupSize, + raw spend cols

more_feature_sets = [
    ("Numerics only (no CryoSleep, no categoricals)", [], NUMERIC_ALL, []),
    ("CryoSleep + Age + TotalSpend (no raw spend cols)", [], ["Age", "TotalSpend"], ["CryoSleep"]),
    ("CryoSleep + Age + raw spend cols (no TotalSpend)", [], ["Age"] + SPEND_COLS, ["CryoSleep"]),
    ("CryoSleep + all numerics", [], NUMERIC_ALL, ["CryoSleep"]),
    ("All categoricals + CryoSleep (no numerics)", ALL_CATEGORICAL, [], ["CryoSleep", "VIP", "IsSolo"]),
    ("All categoricals + numerics (no CryoSleep)", ALL_CATEGORICAL, NUMERIC_ALL, ["VIP", "IsSolo"]),
]

more_results = [evaluate_feature_set(*fs) for fs in more_feature_sets]
results_df = pd.concat([results_df, pd.DataFrame(more_results)], ignore_index=True)
results_df = results_df.sort_values("Accuracy", ascending=False)
results_df[["Experiment", "Accuracy"]]

,Experiment,Accuracy
14,All categoricals + numerics (no CryoSleep),0.797010
0,All engineered features,0.791834
11,CryoSleep + Age + raw spend cols (no TotalSpend),0.780334
9,"Numerics only (no CryoSleep, no categoricals)",0.779183
12,CryoSleep + all numerics,0.778608
1,CryoSleep only,0.713053
2,CryoSleep + HomePlanet,0.713053
10,CryoSleep + Age + TotalSpend (no raw spend cols),0.713053
13,All categoricals + CryoSleep (no numerics),0.711328
3,CryoSleep + HomePlanet + Deck + GroupSize,0.709603


## Collinearity check

Two kinds of redundancy to check before grid search, since collinear features waste model capacity and make coefficients unstable/uninterpretable (less of an issue for trees, more for logistic regression):

1. **Numeric-numeric:** correlation matrix across `Age`, `TotalSpend`, the 5 raw spend columns, `CabinNum`, `GroupSize`. We already know `TotalSpend` is literally the sum of the 5 spend columns — that's deterministic collinearity by construction, not a coincidence, and a heads up that we should pick one or the other for linear models.
2. **Categorical-categorical (and categorical-boolean):** does `HomePlanet` predict `Deck`, and does `CryoSleep` predict `TotalSpend == 0` near-perfectly (we already proved the latter in EDA — that's a hard rule, i.e. perfect collinearity in one direction). We'll quantify both with crosstabs / Cramér's V.

In [5]:
numeric_cols = ["Age", "TotalSpend"] + SPEND_COLS + ["CabinNum", "GroupSize"]
corr_matrix = train[numeric_cols].corr()
print("Numeric correlation matrix:")
print(corr_matrix.round(2))

Numeric correlation matrix:
               Age  TotalSpend  RoomService  FoodCourt  ShoppingMall   Spa  \
Age           1.00        0.18         0.07       0.13          0.03  0.12   
TotalSpend    0.18        1.00         0.23       0.74          0.22  0.59   
RoomService   0.07        0.23         1.00      -0.02          0.05  0.01   
FoodCourt     0.13        0.74        -0.02       1.00         -0.01  0.22   
ShoppingMall  0.03        0.22         0.05      -0.01          1.00  0.01   
Spa           0.12        0.59         0.01       0.22          0.01  1.00   
VRDeck        0.10        0.59        -0.02       0.22         -0.01  0.15   
CabinNum     -0.13       -0.21        -0.01      -0.18          0.00 -0.13   
GroupSize    -0.18        0.01        -0.04       0.03         -0.04  0.02   

              VRDeck  CabinNum  GroupSize  
Age             0.10     -0.13      -0.18  
TotalSpend      0.59     -0.21       0.01  
RoomService    -0.02     -0.01      -0.04  
FoodCourt      

In [6]:
from scipy.stats import chi2_contingency


def cramers_v(col1, col2, df):
    contingency = pd.crosstab(df[col1], df[col2])
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))


cat_pairs = [
    ("HomePlanet", "Deck"),
    ("HomePlanet", "Destination"),
    ("Deck", "Side"),
    ("HomePlanet", "GroupSizeBin"),
]
for a, b in cat_pairs:
    v = cramers_v(a, b, train)
    print(f"Cramer's V({a}, {b}) = {v:.3f}")

print("\nCryoSleep vs (TotalSpend == 0) — already known to be a hard rule from EDA:")
print(pd.crosstab(train["CryoSleep"], train["TotalSpend"] == 0))

Cramer's V(HomePlanet, Deck) = 0.746
Cramer's V(HomePlanet, Destination) = 0.258
Cramer's V(Deck, Side) = 0.040
Cramer's V(HomePlanet, GroupSizeBin) = 0.214

CryoSleep vs (TotalSpend == 0) — already known to be a hard rule from EDA:
TotalSpend  False  True 
CryoSleep               
False        5099    557
True            0   3037


## Re-checking the CryoSleep question with cross-validation

Fair challenge: a 0.5pp gap (0.7970 vs 0.7918) on one 1,739-row validation split could easily be noise — that's well within the kind of split-to-split wobble you'd expect by chance. A single train/val split is a weak basis for "drop this feature" when the gap is this small. Use 5-fold stratified cross-validation on the full training set instead, and look at the *spread* across folds, not just the mean — if the with/without-CryoSleep distributions overlap heavily, the difference isn't real and we should keep CryoSleep on grounds of interpretability and domain logic, not just point-estimate accuracy.

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def cv_scores_for(categorical, numeric, boolean):
    cols = categorical + numeric + boolean
    pipe = build_pipeline(categorical, numeric, boolean)
    scores = cross_val_score(pipe, train[cols], train["Transported"], cv=cv, scoring="accuracy")
    return scores


with_cryo = cv_scores_for(ALL_CATEGORICAL, NUMERIC_ALL, ["CryoSleep", "VIP", "IsSolo"])
without_cryo = cv_scores_for(ALL_CATEGORICAL, NUMERIC_ALL, ["VIP", "IsSolo"])

print("With CryoSleep:    mean={:.4f}  std={:.4f}  folds={}".format(
    with_cryo.mean(), with_cryo.std(), np.round(with_cryo, 4)))
print("Without CryoSleep: mean={:.4f}  std={:.4f}  folds={}".format(
    without_cryo.mean(), without_cryo.std(), np.round(without_cryo, 4)))
print("\nPer-fold difference (with - without):", np.round(with_cryo - without_cryo, 4))
print("Mean difference: {:.4f}  (positive favors keeping CryoSleep)".format((with_cryo - without_cryo).mean()))

With CryoSleep:    mean=0.7908  std=0.0072  folds=[0.7878 0.7901 0.7936 0.8021 0.7802]
Without CryoSleep: mean=0.7850  std=0.0067  folds=[0.7861 0.7844 0.7913 0.7906 0.7727]

Per-fold difference (with - without): [0.0017 0.0058 0.0023 0.0115 0.0075]
Mean difference: 0.0058  (positive favors keeping CryoSleep)


## HomePlanet vs Deck head-to-head

Given Cramér's V = 0.746 between them, keeping both is mostly redundant. Compare each alone (plus numerics + VIP/IsSolo/Side/Destination/GroupSizeBin) via the same 5-fold CV to decide which one earns a place in the final feature set.

In [8]:
OTHER_CATEGORICAL = ["Destination", "Side", "GroupSizeBin"]

homeplanet_only = cv_scores_for(["HomePlanet"] + OTHER_CATEGORICAL, NUMERIC_ALL, ["VIP", "IsSolo"])
deck_only = cv_scores_for(["Deck"] + OTHER_CATEGORICAL, NUMERIC_ALL, ["VIP", "IsSolo"])
both = cv_scores_for(["HomePlanet", "Deck"] + OTHER_CATEGORICAL, NUMERIC_ALL, ["VIP", "IsSolo"])

print("HomePlanet (no Deck): mean={:.4f}  std={:.4f}".format(homeplanet_only.mean(), homeplanet_only.std()))
print("Deck (no HomePlanet): mean={:.4f}  std={:.4f}".format(deck_only.mean(), deck_only.std()))
print("Both together:        mean={:.4f}  std={:.4f}".format(both.mean(), both.std()))

HomePlanet (no Deck): mean=0.7849  std=0.0073
Deck (no HomePlanet): mean=0.7843  std=0.0068
Both together:        mean=0.7850  std=0.0067


## Does adding TotalSpend (on top of raw spend cols) help?

`TotalSpend` is an exact linear sum of the 5 raw spend columns — for a linear model that's pure redundancy. For a tree-based model it could still help by giving the model a ready-made "total spend" split point instead of approximating one across 5 columns. Test both, with vs without `TotalSpend` added to the finalized feature set, via the same 5-fold CV.

In [9]:
FINAL_CATEGORICAL = ["HomePlanet", "Destination", "Side", "GroupSizeBin"]
FINAL_BOOLEAN = ["CryoSleep", "VIP", "IsSolo"]
FINAL_NUMERIC_NO_TOTAL = ["Age", "CabinNum"] + SPEND_COLS
FINAL_NUMERIC_WITH_TOTAL = FINAL_NUMERIC_NO_TOTAL + ["TotalSpend"]


def build_rf_pipeline(categorical, numeric, boolean):
    transformers = []
    if categorical:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical))
    if numeric:
        transformers.append(("num", "passthrough", numeric))  # trees don't need scaling
    if boolean:
        transformers.append(("bool", "passthrough", boolean))
    preprocessor = ColumnTransformer(transformers)
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
    ])


def cv_scores_for_pipeline(pipe, categorical, numeric, boolean):
    cols = categorical + numeric + boolean
    scores = cross_val_score(pipe, train[cols], train["Transported"], cv=cv, scoring="accuracy")
    return scores


for label, numeric_set in [("without TotalSpend", FINAL_NUMERIC_NO_TOTAL), ("with TotalSpend", FINAL_NUMERIC_WITH_TOTAL)]:
    lr_pipe = build_pipeline(FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)
    lr_scores = cv_scores_for_pipeline(lr_pipe, FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)

    rf_pipe = build_rf_pipeline(FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)
    rf_scores = cv_scores_for_pipeline(rf_pipe, FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)

    print(f"[{label}]")
    print(f"  LogisticRegression: mean={lr_scores.mean():.4f}  std={lr_scores.std():.4f}")
    print(f"  RandomForest:       mean={rf_scores.mean():.4f}  std={rf_scores.std():.4f}")

[without TotalSpend]
  LogisticRegression: mean=0.7937  std=0.0102
  RandomForest:       mean=0.7935  std=0.0108


[with TotalSpend]
  LogisticRegression: mean=0.7936  std=0.0104
  RandomForest:       mean=0.7942  std=0.0090


## Does collapsing to TotalSpend-only lose information vs. the 5 raw columns?

Opposite direction from the last check: instead of asking "does adding TotalSpend help," ask "does replacing the 5 raw columns with just TotalSpend hurt?" These aren't symmetric questions — a sum can be redundant *given* the parts (no info gained by adding it) while still being a lossy summary if you *only* keep the sum (info lost by dropping the parts), e.g. if $50 on RoomService vs $50 on Spa predicts differently but both sum to the same total.

In [10]:
FINAL_NUMERIC_TOTAL_ONLY = ["Age", "CabinNum", "TotalSpend"]

for label, numeric_set in [
    ("5 raw spend cols (no TotalSpend)", FINAL_NUMERIC_NO_TOTAL),
    ("TotalSpend only (no raw cols)", FINAL_NUMERIC_TOTAL_ONLY),
]:
    lr_pipe = build_pipeline(FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)
    lr_scores = cv_scores_for_pipeline(lr_pipe, FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)

    rf_pipe = build_rf_pipeline(FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)
    rf_scores = cv_scores_for_pipeline(rf_pipe, FINAL_CATEGORICAL, numeric_set, FINAL_BOOLEAN)

    print(f"[{label}]")
    print(f"  LogisticRegression: mean={lr_scores.mean():.4f}  std={lr_scores.std():.4f}")
    print(f"  RandomForest:       mean={rf_scores.mean():.4f}  std={rf_scores.std():.4f}")

[5 raw spend cols (no TotalSpend)]
  LogisticRegression: mean=0.7937  std=0.0102
  RandomForest:       mean=0.7935  std=0.0108


[TotalSpend only (no raw cols)]
  LogisticRegression: mean=0.7259  std=0.0092
  RandomForest:       mean=0.7195  std=0.0068


## Grid search: LogisticRegression, RandomForest, XGBoost

Finalized feature set from the ablation work above: `CryoSleep`, `HomePlanet`, `Destination`, `Side`, `GroupSizeBin`, `VIP`, `IsSolo`, `Age`, `CabinNum`, and the 5 raw spend columns (no `TotalSpend`, no `Deck`).

Each model gets its own grid, searched with the same 5-fold stratified CV used throughout this notebook so results are comparable to everything above. Trees/boosting don't need scaling, so they get a lighter preprocessing pipeline (one-hot only) than logistic regression.

In [11]:
from xgboost import XGBClassifier
import time

FINAL_COLS = FINAL_CATEGORICAL + FINAL_NUMERIC_NO_TOTAL + FINAL_BOOLEAN
X_final = train[FINAL_COLS]
y_final = train["Transported"].astype(int)

scaled_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), FINAL_CATEGORICAL),
    ("num", StandardScaler(), FINAL_NUMERIC_NO_TOTAL),
    ("bool", "passthrough", FINAL_BOOLEAN),
])

unscaled_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), FINAL_CATEGORICAL),
    ("num", "passthrough", FINAL_NUMERIC_NO_TOTAL),
    ("bool", "passthrough", FINAL_BOOLEAN),
])

grid_specs = {
    "LogisticRegression": (
        Pipeline([("preprocess", scaled_preprocessor),
                  ("model", LogisticRegression(max_iter=2000, random_state=42))]),
        {"model__C": [0.01, 0.1, 1, 10, 100]},
    ),
    "RandomForest": (
        Pipeline([("preprocess", unscaled_preprocessor),
                  ("model", RandomForestClassifier(random_state=42))]),
        {
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_leaf": [1, 2, 4],
        },
    ),
    "XGBoost": (
        Pipeline([("preprocess", unscaled_preprocessor),
                  ("model", XGBClassifier(eval_metric="logloss", random_state=42))]),
        {
            "model__n_estimators": [100, 200, 400],
            "model__max_depth": [3, 5, 7],
            "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        },
    ),
}

grid_results = {}
for name, (pipe, param_grid) in grid_specs.items():
    start = time.time()
    search = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_final, y_final)
    elapsed = time.time() - start
    grid_results[name] = search
    print(f"{name}: best_score={search.best_score_:.4f}  best_params={search.best_params_}  ({elapsed:.1f}s)")

LogisticRegression: best_score=0.7937  best_params={'model__C': 1}  (4.4s)


RandomForest: best_score=0.8032  best_params={'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 400}  (52.6s)


XGBoost: best_score=0.8058  best_params={'model__learning_rate': 0.2, 'model__max_depth': 3, 'model__n_estimators': 100}  (22.1s)


## Expanded grid search

Both RandomForest and XGBoost landed on edge values in the first pass (RF: `n_estimators=400` at the top of its range; XGB: `n_estimators=100` at the bottom, `learning_rate=0.2` at the top). Edge picks mean the grid might have stopped short of the real optimum, so widen ranges in the direction each model pushed:
- XGBoost: extend `n_estimators` down to 50, `learning_rate` up to 0.4, and add `max_depth=2` since 3 was the previous lower edge.
- RandomForest: extend `n_estimators` up to 800; `min_samples_leaf=1` is already the smallest valid value, so no further room to extend there.

In [12]:
expanded_grid_specs = {
    "RandomForest": (
        Pipeline([("preprocess", unscaled_preprocessor),
                  ("model", RandomForestClassifier(random_state=42))]),
        {
            "model__n_estimators": [400, 600, 800],
            "model__max_depth": [10, 15, 20, None],
            "model__min_samples_leaf": [1, 2, 4],
        },
    ),
    "XGBoost": (
        Pipeline([("preprocess", unscaled_preprocessor),
                  ("model", XGBClassifier(eval_metric="logloss", random_state=42))]),
        {
            "model__n_estimators": [50, 100, 200, 400],
            "model__max_depth": [2, 3, 5, 7],
            "model__learning_rate": [0.1, 0.2, 0.3, 0.4],
        },
    ),
}

expanded_results = {}
for name, (pipe, param_grid) in expanded_grid_specs.items():
    start = time.time()
    search = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_final, y_final)
    elapsed = time.time() - start
    expanded_results[name] = search
    print(f"{name}: best_score={search.best_score_:.4f}  best_params={search.best_params_}  ({elapsed:.1f}s)")

RandomForest: best_score=0.8032  best_params={'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__n_estimators': 400}  (198.5s)


XGBoost: best_score=0.8058  best_params={'model__learning_rate': 0.2, 'model__max_depth': 3, 'model__n_estimators': 100}  (22.2s)
